# Análise do Tráfego Benigno — benign_traffic.pcap

Fonte exclusiva: arquivos exportados pelo Wireshark  
Nenhuma informação do artigo ou de fontes externas é utilizada.

Arquivos de entrada:
- `hierarchy.csv` — hierarquia de protocolos  
- `packet_dissection.csv` — log completo de pacotes (2.194.700 linhas)  
- `some_ip_sd_entries.txt` — estatísticas SOME/IP-SD  
- `ipv4_destination_ports.txt` — portas de destino por IP  
- `source_destination_traffic.txt` — volume por IP  
- `ip_protocol_types.txt` — tipos de protocolo IP  
- `packet_lengths.txt` — distribuição de tamanhos  
- `source_ttl.txt` — TTL e destinos por IP fonte  

---

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])

import csv, re, collections
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

BASE = r'c:/Mestrado/SDV_Research/data_exploration/wireshark/bening_traffic/'

BG    = '#0f1117'
PANEL = '#161b2e'
GRID  = '#2a3550'
TEXT  = '#d0d8f0'
A     = '#4e7fff'   # azul
G     = '#7bff9c'   # verde
T     = '#7bffd9'   # teal
Y     = '#ffd97b'   # amarelo
P     = '#c07bff'   # roxo
O     = '#ff9d7b'   # laranja
R     = '#ff7b7b'   # vermelho

LB = dict(paper_bgcolor=BG, plot_bgcolor=PANEL,
          font=dict(color=TEXT, family='monospace'),
          title_font=dict(color=TEXT, size=15))

def ha(h, a=1.0):
    r,g,b = int(h[1:3],16),int(h[3:5],16),int(h[5:7],16)
    return f'rgba({r},{g},{b},{a})'

_first = True
def show(fig):
    global _first
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn' if _first else False)))
    _first = False

print('OK')

OK


## 1. Hierarquia de Protocolos
**Fonte: `hierarchy.csv`**

In [2]:
# Dados diretos do hierarchy.csv
# Linhas "end packets" = pacotes que terminam naquele protocolo (sem payload de nível superior)
hier = pd.DataFrame([
    {'Protocolo': 'SOME/IP (TCP)',     'Pacotes':  1086986, 'Bytes':  42822984},
    {'Protocolo': 'TCP (ctrl/ACK)',    'Pacotes':  1085100, 'Bytes':  34723480},
    {'Protocolo': 'SOME/IP-SD (UDP)', 'Pacotes':    18695, 'Bytes':    676184},
    {'Protocolo': 'SOME/IP (UDP)',     'Pacotes':     3021, 'Bytes':    110332},
    {'Protocolo': 'ARP',               'Pacotes':      876, 'Bytes':     24528},
    {'Protocolo': 'IGMPv3',            'Pacotes':       22, 'Bytes':       352},
])
hier['% Pacotes'] = (hier['Pacotes'] / hier['Pacotes'].sum() * 100).round(2)

colors = [G, A, Y, T, P, O]
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Pacotes por protocolo', 'Bytes por protocolo'],
    specs=[[{'type':'pie'},{'type':'pie'}]])
fig.add_trace(go.Pie(labels=hier['Protocolo'], values=hier['Pacotes'],
    hole=0.4, marker_colors=colors, textinfo='label+percent',
    hovertemplate='%{label}<br>%{value:,}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Pie(labels=hier['Protocolo'], values=hier['Bytes'],
    hole=0.4, marker_colors=colors, textinfo='label+percent',
    hovertemplate='%{label}<br>%{value:,} bytes<extra></extra>'), row=1, col=2)
fig.update_layout(**LB, title='hierarchy.csv — 2.194.700 frames capturados', height=420)
show(fig)
print(hier.to_string(index=False))

       Protocolo  Pacotes    Bytes  % Pacotes
   SOME/IP (TCP)  1086986 42822984      49.53
  TCP (ctrl/ACK)  1085100 34723480      49.44
SOME/IP-SD (UDP)    18695   676184       0.85
   SOME/IP (UDP)     3021   110332       0.14
             ARP      876    24528       0.04
          IGMPv3       22      352       0.00


## 2. IPs da Rede — Volume de Tráfego
**Fonte: `source_destination_traffic.txt`**

In [3]:
# Valores extraídos diretamente de source_destination_traffic.txt
ips = pd.DataFrame([
    {'IP': '172.18.0.10', 'Enviados': 495238, 'Recebidos': 493742},
    {'IP': '172.18.0.5',  'Enviados': 422160, 'Recebidos': 420697},
    {'IP': '172.18.0.2',  'Enviados': 321166, 'Recebidos': 321531},
    {'IP': '172.18.0.7',  'Enviados': 203836, 'Recebidos': 204254},
    {'IP': '172.18.0.4',  'Enviados': 181453, 'Recebidos': 180501},
    {'IP': '172.18.0.9',  'Enviados': 180282, 'Recebidos': 180429},
    {'IP': '172.18.0.6',  'Enviados': 159542, 'Recebidos': 159710},
    {'IP': '172.18.0.8',  'Enviados': 130765, 'Recebidos': 131151},
    {'IP': '172.18.0.3',  'Enviados':  99382, 'Recebidos':  99523},
])

fig = go.Figure()
fig.add_trace(go.Bar(name='Enviados',   x=ips['IP'], y=ips['Enviados'],
    marker_color=A, hovertemplate='%{x}<br>Enviados: %{y:,}<extra></extra>'))
fig.add_trace(go.Bar(name='Recebidos', x=ips['IP'], y=ips['Recebidos'],
    marker_color=ha(A,0.5), hovertemplate='%{x}<br>Recebidos: %{y:,}<extra></extra>'))
fig.update_layout(**LB, title='source_destination_traffic.txt — pacotes por IP',
    barmode='group', height=420,
    xaxis=dict(title='IP', gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID))
show(fig)

print('9 endereços IPv4 únicos: 172.18.0.2 a 172.18.0.10')
print('Enviados ≈ Recebidos para todos os IPs — sem assimetria acentuada no total.')

9 endereços IPv4 únicos: 172.18.0.2 a 172.18.0.10
Enviados ≈ Recebidos para todos os IPs — sem assimetria acentuada no total.


## 3. Distribuição de Tamanho de Pacotes
**Fonte: `packet_lengths.txt`**

In [4]:
# Valores diretos de packet_lengths.txt
# Count total reportado: 835.066 (subconjunto filtrado no Wireshark, não o total)
lengths = pd.DataFrame([
    {'Faixa': '40–79 bytes',   'Count': 414157, 'Min': 42,  'Max': 78,  'Avg': 65.98},
    {'Faixa': '80–159 bytes',  'Count': 420896, 'Min': 82,  'Max': 154, 'Avg': 105.07},
    {'Faixa': '160–319 bytes', 'Count':     13, 'Min': 162, 'Max': 178, 'Avg': 165.69},
])

fig = go.Figure(go.Bar(
    x=lengths['Faixa'], y=lengths['Count'],
    marker_color=[A, G, Y],
    text=[f'{v:,}' for v in lengths['Count']],
    textposition='outside',
    hovertemplate='%{x}<br>Count: %{y:,}<extra></extra>',
))
fig.update_layout(**LB, title='packet_lengths.txt — distribuição por faixa de tamanho',
    xaxis=dict(title='Faixa de tamanho (bytes)', gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID), height=380)
show(fig)

print('Estatísticas reportadas no packet_lengths.txt:')
print(f'  Total amostrado : 835.066 pacotes')
print(f'  Média           : 85,68 bytes')
print(f'  Mínimo          : 42 bytes')
print(f'  Máximo          : 178 bytes  (na faixa 160–319, apenas 13 pacotes)')
print(f'  40–79  bytes    : 49,60% ({414157:,} pacotes)')
print(f'  80–159 bytes    : 50,40% ({420896:,} pacotes)')
print(f'  160+   bytes    :  0,00% ({13} pacotes)')

Estatísticas reportadas no packet_lengths.txt:
  Total amostrado : 835.066 pacotes
  Média           : 85,68 bytes
  Mínimo          : 42 bytes
  Máximo          : 178 bytes  (na faixa 160–319, apenas 13 pacotes)
  40–79  bytes    : 49,60% (414,157 pacotes)
  80–159 bytes    : 50,40% (420,896 pacotes)
  160+   bytes    :  0,00% (13 pacotes)


## 4. SOME/IP-SD — Service Discovery
**Fonte: `some_ip_sd_entries.txt`**

In [5]:
# Extraído de some_ip_sd_entries.txt
# Termos "Offer", "Find", "Subscribe", "SubscribeAck" vêm do próprio log

sd = pd.DataFrame([
    {'IP': '172.18.0.10', 'Offer': 748, 'SubscribeAck': 2990, 'Subscribe': 0,    'Find': 0},
    {'IP': '172.18.0.5',  'Offer': 748, 'SubscribeAck': 2984, 'Subscribe': 0,    'Find': 0},
    {'IP': '172.18.0.4',  'Offer': 748, 'SubscribeAck': 2241, 'Subscribe': 0,    'Find': 0},
    {'IP': '172.18.0.2',  'Offer': 0,   'SubscribeAck': 0,    'Subscribe': 2229, 'Find': 3},
    {'IP': '172.18.0.9',  'Offer': 0,   'SubscribeAck': 0,    'Subscribe': 2245, 'Find': 12},
    {'IP': '172.18.0.6',  'Offer': 0,   'SubscribeAck': 0,    'Subscribe': 1497, 'Find': 8},
    {'IP': '172.18.0.7',  'Offer': 0,   'SubscribeAck': 0,    'Subscribe': 748,  'Find': 8},
    {'IP': '172.18.0.8',  'Offer': 0,   'SubscribeAck': 0,    'Subscribe': 747,  'Find': 4},
    {'IP': '172.18.0.3',  'Offer': 0,   'SubscribeAck': 0,    'Subscribe': 749,  'Find': 4},
])

fig = go.Figure()
for col, color in [('Find', Y), ('Offer', G), ('Subscribe', A), ('SubscribeAck', T)]:
    fig.add_trace(go.Bar(name=col, x=sd['IP'], y=sd[col], marker_color=color,
        hovertemplate=f'{col}: %{{y:,}}<extra></extra>'))
fig.update_layout(**LB, title='some_ip_sd_entries.txt — mensagens SD por IP',
    barmode='stack', height=420,
    xaxis=dict(title='IP', gridcolor=GRID),
    yaxis=dict(title='Mensagens', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID))
show(fig)

print('IPs que enviam Offer (têm o serviço):')
print('  172.18.0.10 — Offer Service 0x1001')
print('  172.18.0.5  — Offer Service 0x1002')
print('  172.18.0.4  — Offer Service 0x1003')
print()
print('IPs que enviam Subscribe (solicitam o serviço):')
print('  172.18.0.2  — Subscribe 0x1001 + 0x1002 + 0x1003')
print('  172.18.0.9  — Subscribe 0x1001 + 0x1002 + 0x1003')
print('  172.18.0.6  — Subscribe 0x1002 + 0x1003')
print('  172.18.0.7  — Subscribe 0x1001')
print('  172.18.0.8  — Subscribe 0x1002')
print('  172.18.0.3  — Subscribe 0x1001')
print()
print('Fonte: some_ip_sd_entries.txt (Subscribe Eventgroup por IP e Service ID)')

IPs que enviam Offer (têm o serviço):
  172.18.0.10 — Offer Service 0x1001
  172.18.0.5  — Offer Service 0x1002
  172.18.0.4  — Offer Service 0x1003

IPs que enviam Subscribe (solicitam o serviço):
  172.18.0.2  — Subscribe 0x1001 + 0x1002 + 0x1003
  172.18.0.9  — Subscribe 0x1001 + 0x1002 + 0x1003
  172.18.0.6  — Subscribe 0x1002 + 0x1003
  172.18.0.7  — Subscribe 0x1001
  172.18.0.8  — Subscribe 0x1002
  172.18.0.3  — Subscribe 0x1001

Fonte: some_ip_sd_entries.txt (Subscribe Eventgroup por IP e Service ID)


## 5. SOME/IP — Dados: Service IDs, Method IDs e IP Fonte
**Fonte: `packet_dissection.csv` (2.194.700 linhas, processado completo)**

In [6]:
# Resultado do processamento completo do packet_dissection.csv
# Campos disponíveis no Info: Service ID, Method ID, Length (campo SOME/IP)
# Message Type NÃO aparece no campo Info exportado — requer dissecção completa

flows = pd.DataFrame([
    {'Service ID': '0x1001', 'Method ID': '0x0002', 'IP Fonte': '172.18.0.10', 'Pacotes': 203504},
    {'Service ID': '0x1002', 'Method ID': '0x0003', 'IP Fonte': '172.18.0.5',  'Pacotes': 130403},
    {'Service ID': '0x1001', 'Method ID': '0x0001', 'IP Fonte': '172.18.0.10', 'Pacotes': 129832},
    {'Service ID': '0x1002', 'Method ID': '0x0001', 'IP Fonte': '172.18.0.5',  'Pacotes': 129776},
    {'Service ID': '0x1002', 'Method ID': '0x0002', 'IP Fonte': '172.18.0.5',  'Pacotes':  98826},
    {'Service ID': '0x1001', 'Method ID': '0x0003', 'IP Fonte': '172.18.0.10', 'Pacotes':  98771},
    {'Service ID': '0x1003', 'Method ID': '0x0001', 'IP Fonte': '172.18.0.4',  'Pacotes':  59690},
    {'Service ID': '0x1002', 'Method ID': '0x0004', 'IP Fonte': '172.18.0.5',  'Pacotes':  59414},
    {'Service ID': '0x1003', 'Method ID': '0x0002', 'IP Fonte': '172.18.0.4',  'Pacotes':  59385},
    {'Service ID': '0x1003', 'Method ID': '0x0003', 'IP Fonte': '172.18.0.4',  'Pacotes':  59384},
    {'Service ID': '0x1001', 'Method ID': '0x0004', 'IP Fonte': '172.18.0.10', 'Pacotes':  59383},
    {'Service ID': '0x1001', 'Method ID': '0x0001', 'IP Fonte': '172.18.0.2',  'Pacotes':    568},
    {'Service ID': '0x1003', 'Method ID': '0x0001', 'IP Fonte': '172.18.0.2',  'Pacotes':    541},
    {'Service ID': '0x1002', 'Method ID': '0x0001', 'IP Fonte': '172.18.0.2',  'Pacotes':    530},
])

svc_color = {'0x1001': G, '0x1002': T, '0x1003': Y}

fig = go.Figure(go.Bar(
    x=[f"{r['Service ID']}\n{r['Method ID']}\n{r['IP Fonte']}" for _,r in flows.iterrows()],
    y=flows['Pacotes'],
    marker_color=[svc_color[s] for s in flows['Service ID']],
    hovertemplate='Service %{x}<br>%{y:,} pacotes<extra></extra>',
))
fig.update_layout(**LB,
    title='packet_dissection.csv — pacotes SOME/IP por (Service ID, Method ID, IP Fonte)',
    xaxis=dict(title='Service / Method / Fonte', tickangle=45, gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID, type='log'),
    height=480, showlegend=False)
show(fig)

print(flows.to_string(index=False))
print()
print('Observação: 172.18.0.2 envia Method 0x0001 para os 3 serviços (1.639 pacotes no total).')
print('O significado de Method 0x0001 vs 0x0002/0x0003/0x0004 não é inferível apenas do CSV.')
print('Message Type (REQUEST/NOTIFICATION/etc.) requer dissecção completa por pacote.')

Service ID Method ID    IP Fonte  Pacotes
    0x1001    0x0002 172.18.0.10   203504
    0x1002    0x0003  172.18.0.5   130403
    0x1001    0x0001 172.18.0.10   129832
    0x1002    0x0001  172.18.0.5   129776
    0x1002    0x0002  172.18.0.5    98826
    0x1001    0x0003 172.18.0.10    98771
    0x1003    0x0001  172.18.0.4    59690
    0x1002    0x0004  172.18.0.5    59414
    0x1003    0x0002  172.18.0.4    59385
    0x1003    0x0003  172.18.0.4    59384
    0x1001    0x0004 172.18.0.10    59383
    0x1001    0x0001  172.18.0.2      568
    0x1003    0x0001  172.18.0.2      541
    0x1002    0x0001  172.18.0.2      530

Observação: 172.18.0.2 envia Method 0x0001 para os 3 serviços (1.639 pacotes no total).
O significado de Method 0x0001 vs 0x0002/0x0003/0x0004 não é inferível apenas do CSV.
Message Type (REQUEST/NOTIFICATION/etc.) requer dissecção completa por pacote.


## 6. Frame Sizes por (Service ID, Method ID)
**Fonte: `packet_dissection.csv` — campo Length**

In [7]:
# Tamanhos de frame (Ethernet) únicos observados por combinação Service/Method
frame_data = pd.DataFrame([
    {'Service': '0x1001', 'Method': '0x0001', 'Tamanhos observados (bytes)': '62, 82, 90, 98, 106, 114, 122, 130, 138, 146, 154, 162, 170, 178, 186', 'Tipo': 'variável'},
    {'Service': '0x1001', 'Method': '0x0002', 'Tamanhos observados (bytes)': '106',                                                                     'Tipo': 'fixo'},
    {'Service': '0x1001', 'Method': '0x0003', 'Tamanhos observados (bytes)': '98',                                                                      'Tipo': 'fixo'},
    {'Service': '0x1001', 'Method': '0x0004', 'Tamanhos observados (bytes)': '106',                                                                     'Tipo': 'fixo'},
    {'Service': '0x1002', 'Method': '0x0001', 'Tamanhos observados (bytes)': '58, 62, 66, 74, 82, 90, 98, 106, 114',                                   'Tipo': 'variável'},
    {'Service': '0x1002', 'Method': '0x0002', 'Tamanhos observados (bytes)': '114',                                                                     'Tipo': 'fixo'},
    {'Service': '0x1002', 'Method': '0x0003', 'Tamanhos observados (bytes)': '98',                                                                      'Tipo': 'fixo'},
    {'Service': '0x1002', 'Method': '0x0004', 'Tamanhos observados (bytes)': '98',                                                                      'Tipo': 'fixo'},
    {'Service': '0x1003', 'Method': '0x0001', 'Tamanhos observados (bytes)': '58, 62, 66, 74, 82, 90, 98, 106, 122',                                   'Tipo': 'variável'},
    {'Service': '0x1003', 'Method': '0x0002', 'Tamanhos observados (bytes)': '122',                                                                     'Tipo': 'fixo'},
    {'Service': '0x1003', 'Method': '0x0003', 'Tamanhos observados (bytes)': '106',                                                                     'Tipo': 'fixo'},
])

tipo_color = {'fixo': G, 'variável': O}
fig = go.Figure(go.Table(
    header=dict(
        values=['Service ID', 'Method ID', 'Frame sizes (bytes)', 'Tamanho'],
        fill_color=PANEL, font=dict(color=A, size=12), align='left'),
    cells=dict(
        values=[frame_data['Service'], frame_data['Method'],
                frame_data['Tamanhos observados (bytes)'], frame_data['Tipo']],
        fill_color=[[BG if i%2==0 else PANEL for i in range(len(frame_data))]]*4,
        font=dict(color=[[tipo_color[t] if col==3 else TEXT
                          for t in frame_data['Tipo']] for col in range(4)], size=11),
        align='left')
))
fig.update_layout(**LB, title='Frame sizes únicos por (Service ID, Method ID) — packet_dissection.csv', height=420)
show(fig)

print('Method 0x0001: frame size VARIÁVEL para todos os serviços')
print('Methods 0x0002/0x0003/0x0004: frame size FIXO por serviço')
print()
print('Nota: não é possível determinar pelo CSV exportado se os tamanhos variáveis')
print('do Method 0x0001 refletem tipo de mensagem diferente ou segmentação TCP.')

Method 0x0001: frame size VARIÁVEL para todos os serviços
Methods 0x0002/0x0003/0x0004: frame size FIXO por serviço

Nota: não é possível determinar pelo CSV exportado se os tamanhos variáveis
do Method 0x0001 refletem tipo de mensagem diferente ou segmentação TCP.


## 7. Portas TCP por IP de Destino
**Fonte: `ipv4_destination_ports.txt`**

In [8]:
# Extraído de ipv4_destination_ports.txt
# Apenas portas com volume expressivo são listadas
# A identificação de protocolo (SOME/IP vs outro) NÃO vem deste arquivo

ports = pd.DataFrame([
    {'IP Destino': '172.18.0.10', 'Protocolo': 'TCP', 'Porta': 30501, 'Pacotes': 490184},
    {'IP Destino': '172.18.0.5',  'Protocolo': 'TCP', 'Porta': 30502, 'Pacotes': 417183},
    {'IP Destino': '172.18.0.4',  'Protocolo': 'TCP', 'Porta': 30503, 'Pacotes': 177719},
    {'IP Destino': '172.18.0.2',  'Protocolo': 'TCP', 'Porta': 41621, 'Pacotes': 129405},
    {'IP Destino': '172.18.0.2',  'Protocolo': 'TCP', 'Porta': 36809, 'Pacotes': 129364},
    {'IP Destino': '172.18.0.9',  'Protocolo': 'TCP', 'Porta': 38753, 'Pacotes':  59415},
    {'IP Destino': '172.18.0.9',  'Protocolo': 'TCP', 'Porta': 41827, 'Pacotes':  59385},
    {'IP Destino': '172.18.0.9',  'Protocolo': 'TCP', 'Porta': 34133, 'Pacotes':  59384},
    {'IP Destino': '172.18.0.6',  'Protocolo': 'TCP', 'Porta': 36369, 'Pacotes':  98827},
    {'IP Destino': '172.18.0.6',  'Protocolo': 'TCP', 'Porta': 37291, 'Pacotes':  59386},
    {'IP Destino': '172.18.0.7',  'Protocolo': 'TCP', 'Porta': 32959, 'Pacotes': 203505},
    {'IP Destino': '172.18.0.8',  'Protocolo': 'TCP', 'Porta': 34501, 'Pacotes': 130404},
    {'IP Destino': '172.18.0.3',  'Protocolo': 'TCP', 'Porta': 38743, 'Pacotes':  98774},
    {'IP Destino': '172.18.0.2',  'Protocolo': 'TCP', 'Porta': 33609, 'Pacotes':  59150},
    # UDP
    {'IP Destino': '224.244.224.245', 'Protocolo': 'UDP', 'Porta': 30490, 'Pacotes': 2264},
    {'IP Destino': '172.18.0.5',  'Protocolo': 'UDP', 'Porta': 30490, 'Pacotes': 2984},
    {'IP Destino': '172.18.0.10', 'Protocolo': 'UDP', 'Porta': 30490, 'Pacotes': 2990},
    {'IP Destino': '172.18.0.4',  'Protocolo': 'UDP', 'Porta': 30490, 'Pacotes': 2241},
])

# Destacar as portas que recebem maior volume (as fixas)
top = ports.nlargest(8, 'Pacotes')
fig = go.Figure(go.Bar(
    x=[f"{r['IP Destino']}:{r['Porta']}" for _,r in top.iterrows()],
    y=top['Pacotes'],
    marker_color=[G if r['Porta'] in (30501,30502,30503) else A
                  for _,r in top.iterrows()],
    text=[f"{v:,}" for v in top['Pacotes']],
    textposition='outside',
    hovertemplate='%{x}<br>%{y:,} pacotes<extra></extra>',
))
fig.update_layout(**LB, title='ipv4_destination_ports.txt — top 8 pares IP:Porta por volume TCP',
    xaxis=dict(title='IP:Porta', tickangle=30, gridcolor=GRID),
    yaxis=dict(title='Pacotes TCP recebidos', gridcolor=GRID),
    height=420, showlegend=False)
show(fig)

print('Portas fixas com maior volume de recepção TCP:')
print('  172.18.0.10 : porta 30501  —  490.184 pacotes')
print('  172.18.0.5  : porta 30502  —  417.183 pacotes')
print('  172.18.0.4  : porta 30503  —  177.719 pacotes')
print()
print('Portas efêmeras dos clientes (exemplos observados):')
print('  172.18.0.7  :32959  172.18.0.6  :36369  172.18.0.6  :37291')
print('  172.18.0.8  :34501  172.18.0.3  :38743  172.18.0.9  :38753')
print()
print('UDP porta 30490: trafego SD (SOME/IP-SD) — multicast 224.244.224.245 e unicast')
print()
print('Nota: a associação entre porta e protocolo SOME/IP vem do Wireshark (hierarchy.csv),')
print('não deste arquivo. ipv4_destination_ports.txt reporta apenas contagem por porta.')

Portas fixas com maior volume de recepção TCP:
  172.18.0.10 : porta 30501  —  490.184 pacotes
  172.18.0.5  : porta 30502  —  417.183 pacotes
  172.18.0.4  : porta 30503  —  177.719 pacotes

Portas efêmeras dos clientes (exemplos observados):
  172.18.0.7  :32959  172.18.0.6  :36369  172.18.0.6  :37291
  172.18.0.8  :34501  172.18.0.3  :38743  172.18.0.9  :38753

UDP porta 30490: trafego SD (SOME/IP-SD) — multicast 224.244.224.245 e unicast

Nota: a associação entre porta e protocolo SOME/IP vem do Wireshark (hierarchy.csv),
não deste arquivo. ipv4_destination_ports.txt reporta apenas contagem por porta.


## 8. Resumo Factual
**Somente o que está nos logs do Wireshark**

In [9]:
print('=' * 68)
print('RESUMO — benign_traffic.pcap — apenas dados observados')
print('=' * 68)

facts = [
    ('CAPTURA', [
        ('Total de frames',    '2.194.700'),
        ('Total de bytes',     '188.216.628 (~188 MB)'),
        ('Taxa média',         '1.469 pkt/ms (1.470 pkt/s)'),
        ('Burst máximo',       '1.980 pkt/s'),
    ]),
    ('PROTOCOLOS (hierarchy.csv)', [
        ('TCP',                '99,01% dos pacotes IP'),
        ('UDP',                '0,99% — apenas SOME/IP-SD'),
        ('SOME/IP (TCP)',      '1.086.986 pacotes / 22,75% dos bytes'),
        ('SOME/IP-SD (UDP)',   '18.695 pacotes'),
    ]),
    ('IPs (9 endereços únicos)', [
        ('Enviam Offer SD',    '172.18.0.10 (0x1001) / .5 (0x1002) / .4 (0x1003)'),
        ('Enviam Subscribe SD','172.18.0.2, .3, .6, .7, .8, .9'),
        ('Maior emissor',      '172.18.0.10 — 495.238 pkts enviados'),
        ('TTL',                '64 em todos os pacotes IP unicast'),
    ]),
    ('TAMANHO DE PACOTE (packet_lengths.txt)', [
        ('Mínimo',             '42 bytes'),
        ('Máximo',             '178 bytes'),
        ('Média',              '85,68 bytes'),
        ('Faixa 40–79 bytes',  '49,60% — ACKs, ARP, SYN'),
        ('Faixa 80–159 bytes', '50,40% — SOME/IP data + SD'),
    ]),
    ('SOME/IP (packet_dissection.csv)', [
        ('Service IDs',        '0x1001  0x1002  0x1003'),
        ('Method IDs',         '0x0001  0x0002  0x0003  0x0004'),
        ('Method 0x0001',      'frame size VARIÁVEL (58–186 bytes) — enviado por offerer + 172.18.0.2'),
        ('Methods 0x0002–0004','frame size FIXO por service'),
        ('Porta SD (UDP)',      '30490'),
        ('Portas fixas (TCP)',  '30501 (.10) / 30502 (.5) / 30503 (.4)'),
    ]),
    ('NÃO OBSERVADO NOS LOGS', [
        ('Nomes de serviço',   'GPS/IMU/VDE — não aparecem em nenhum arquivo Wireshark'),
        ('Message Type',       'não exportado no campo Info do packet_dissection.csv'),
        ('Significado métodos','0x0001 vs 0x0002 etc. — requer dissecção completa'),
    ]),
]

for section, items in facts:
    print(f'\n  [{section}]')
    for k, v in items:
        print(f'    {k:<28}: {v}')

RESUMO — benign_traffic.pcap — apenas dados observados

  [CAPTURA]
    Total de frames             : 2.194.700
    Total de bytes              : 188.216.628 (~188 MB)
    Taxa média                  : 1.469 pkt/ms (1.470 pkt/s)
    Burst máximo                : 1.980 pkt/s

  [PROTOCOLOS (hierarchy.csv)]
    TCP                         : 99,01% dos pacotes IP
    UDP                         : 0,99% — apenas SOME/IP-SD
    SOME/IP (TCP)               : 1.086.986 pacotes / 22,75% dos bytes
    SOME/IP-SD (UDP)            : 18.695 pacotes

  [IPs (9 endereços únicos)]
    Enviam Offer SD             : 172.18.0.10 (0x1001) / .5 (0x1002) / .4 (0x1003)
    Enviam Subscribe SD         : 172.18.0.2, .3, .6, .7, .8, .9
    Maior emissor               : 172.18.0.10 — 495.238 pkts enviados
    TTL                         : 64 em todos os pacotes IP unicast

  [TAMANHO DE PACOTE (packet_lengths.txt)]
    Mínimo                      : 42 bytes
    Máximo                      : 178 bytes
    Média 